# Wikidata Categorization - Pliny's Wall of Fame

This code models the Wikidata property values attached to people cited in books 2-6 of the _Naturalis Historia_ by Pliny the Elder. 

The code is made available together with the paper _Pliny’s “Wall of Fame”: A Digital Semantic Approach to Citations and Character Networks in Books 2-6 of the Naturalis Historia_, published in DH Benelux Journal.

The notebook is divided into five sections:
- Authors
- Citizenship
- Language
- Occupation
- Chronology

Use the file **"nh_extended_withdates.csv"**: this already contains the 'is_author_index' correct value and the chronological dates extracted from Evelien de Graaf's dataset.


## Importing

In [290]:
import pandas as pd

## Loading data

In [292]:
df = pd.read_csv('nh_extended_withdates.csv')

C:\Users\u0172188\AppData\Local\Temp\ipykernel_18688\2964155296.py:1: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('nh_extended_withdates.csv')


## Adding manual information

In [349]:
# Kandake (Q615170): manually adjusting to human (on wikidata it is noble title)
df.loc[df['wikidata_qid'] == 'Q615170', 'is_human'] = True

# L. Aemilius Paulus: Latin language and Roman citizenship (see paper for explanation)
mask = df['wikidata_qid'] == 'unspec'
df.loc[mask, ['language_group']] = [ 'Latin']
df.loc[mask, ['citizenship_group']] = ['Roma']
df.loc[mask, ['is_human']] = True

## Authors

This code assigns the persons to the author category, according to their Wikidata occupation values

In [296]:
df_authors = df.copy()

#defining authors occupation label (see appendix B of the paper)
authors_occupation_labels = ["Q36180", "Q4964182", "Q49757", "Q11063", "Q170790", "Q11063", "Q864503", "Q12859263", "Q201788", "Q11900058",
                            "Q185351", "Q816349", "Q39631", "Q16868675", "Q294126", "Q901402", "Q42973" ]

#removing false positives from the results: this list has been compiled after a manual inspection of the data
not_authors_wiki_id = ["Q1413", "Q1048", "Q191039", "Q297783", "Q2933380", "Q380453", "Q2674522", "Q365148", "Q123058247"]

In [297]:
def is_author(occupation_id, authors_occupation_labels, wikidata_id, human_label):
    """
    Check if occupation label contains an author id
    """

    #we work only with humans
    if human_label == False:
        return "not_human"
    
    elif human_label == True:  
        
        if pd.isna(occupation_id):
            return "unspec"

        if any(not_author in str(wikidata_id) for not_author in not_authors_wiki_id):
            return False
    
        if any(author_id in str(occupation_id) for author_id in authors_occupation_labels):
            return True
        else:
            return False


df_authors['is_author_wikidata'] = df_authors.apply(
    lambda row: is_author(row['occupation_id'], authors_occupation_labels, row['wikidata_qid'], row['is_human']),
                         axis = 1)


## Citizenship

We divide the human entities into five groups:
- Roma
- Graecia
- Asia
- Africa
- Germania


In [300]:
#definition of citizenship groups
rome = ['Q1747689', 'Q9160295']
greece = ['Q1568260','Q169460', 'Q13580795', 'Q11936633', 'Q833665', 'Q844930', 'Q83958', 'Q1233764', 'Q5843680',
         'Q244796', 'Q11266977']
asia = ['Q93180', 'Q389688', 'Q22647',  'Q2022162',   'Q1986139', 'Q1062946', 'Q620765', 
        'Q7380487', 'Q16000109', 'Q163490'] #moved Sidon and Judaea
africa = ['Q2320005',  'Q2429397', 'Q6343', 'Q309272', 'Q102679', 'Q11768'  ]

After manually checking, we define some names in the labels to be assigned directly to a specific group

In [302]:
asia_name = ['Charax', 'Macedon']
africa_name = [ "Nechepso", "Alexandria", "Mauretania",  "Lathyros", "Ceraunus", "Naucratis" ]

In [303]:
citiz_df = df_authors.copy()

As described in the paper, first we use the citizenship property values. If they are not available, we check the languages

In [305]:
def categorize_by_region(value_citiz, value_lang, qid, wikidata_label):

    #check citizenship

    if pd.notna(value_citiz):
        value_str = str(value_citiz)
    
        if any(country in value_str for country in rome):
            return "Roma"
        elif any(country in value_str for country in greece):
            return "Graecia"
        elif any(country in value_str for country in asia):
            return "Asia"
        elif any(country in value_str for country in africa):
            return "Africa"


    #check language
    
    if pd.notna(value_lang):
        if value_lang in ['Q397', 'Q36653']: #latin and oscan
            return "Roma"
        elif value_lang == "Q35497": #if ancient greek
            if any(string in wikidata_label for string in africa_name):
                return "Africa"
            elif any(string in wikidata_label for string in asia_name):
                return "Asia"
            else:
                return "Graecia"
        elif value_lang in ["Q11059", "Q35225", "Q8785"]: #sanskrit, old persian, armenian
            return "Asia"
        else:
            return "Other"
            
    #for new citizenships, added later to Wikidata
    if pd.notna(value_citiz):
        return "unclassified"
#manual assignment

    else:
        if qid in ["Q2825506", "Q2347534", "Q353141", "Q615170" ]:
            return "Africa"
        elif qid in ["Q299551", "Q741999", "Q957489", "Q1519202", "Q103960", "Q5336564", "Q312446", "Q62571419" ]:
            return "Asia"
        elif qid  in ["Q16190879", "Q16190879", "Q88976791", "Q11955354", "Q5279393" ]:
            return "Graecia"
        elif qid == "Q1606500":
            return "Germania"
        elif qid == "unspec":
            return "Roma" #we know for sure the individual called L. Aemilius Paulus was a Roman, even if we cannot identify him (see fn. 13)
        elif pd.isna(qid):
            return "not_person"
        else:
            return "not_human"


citiz_df['citizenship_group'] = citiz_df.apply(
    lambda row: categorize_by_region(row['citizenship_id'], row['language_id'], row['wikidata_qid'], row['Wikidata_label']), 
    axis=1
)

print(citiz_df['citizenship_group'].value_counts())

citizenship_group
not_person    69278
Roma            462
not_human       306
Graecia         303
Asia             81
Africa           38
Germania          1
Name: count, dtype: int64


## Language


For the language, I will divide the entities in three groups:
- Latin
- Greek
- Other

In [308]:
#defining the groups
latin_lang = ["Q397"] 
greek_lang = ["Q35497", "Q406373", "Q107358"] 
other_lang = ["Q535958", "Q36653", "Q36095", "Q35225", "Q50868", "Q946997", "Q35470", "Q11059", "Q8785"]

In [309]:
language_df = citiz_df.copy()

In [310]:

def categorize_by_language(value_citiz, value_lang, qid):

    #checking the language value first
    if pd.notna(value_lang):
        value_str = str(value_lang)
    

    if pd.notna(value_lang):
        if  "Q397" in value_str:
            return "Latin"
        elif any(lang in value_str for lang in greek_lang):
            return "Greek"
        else:
            return "Other"

    #if not available, look at the citizenship: if Ancient Rome > Latin 

    elif pd.notna(value_citiz):
        if value_citiz == "Q1747689":
            return "Latin"
        else:
            return "Other"

    #manual assignment

    else:
        if qid in ["Q2825506", "Q2347534", "Q353141", "Q615170", "Q299551", "Q741999", "Q957489", "Q1519202",
                   "Q103960", "Q5336564", "Q312446", "Q62571419", "Q1606500" ]:
            return "Other"
        elif qid  in ["Q16190879", "Q16190879", "Q88976791", "Q11955354", "Q5279393" ]:
            return "Greek"
        elif qid == "unspec":
            return "Latin" #we know for sure the individual called L. Aemilius Paulus was a Roman and spoke Latin, even if we cannot identify him (see fn. 13)
        elif pd.isna(qid):
            return "not_person"
        else:
            return "not_human"


language_df['language_group'] = language_df.apply(
    lambda row: categorize_by_language(row['citizenship_id'], row['language_id'], row['wikidata_qid']), 
    axis=1
)

print(language_df['language_group'].value_counts())


language_group
not_person    69278
Latin           437
Greek           386
not_human       306
Other            62
Name: count, dtype: int64


## Occupation

As described in the paper, we focus now only on **authors**. For this reason, it is necessary to start from the version of the dataframe including the 'is_author_index' value, which includes only correct data (does not include false negatives).

For this case, we follow a different procedure than before: we first assign boolean values for each defined provisional category (_def categorize_occupations_); then we assign them to the final one, using the boolean values (_def assign_unique_occupation_).

In [313]:
#defining groups
politician = ['Q82955', 'Q47064', 'Q1402561', 'Q116', 'Q1097498', 'Q2304859', 'Q3409374', 'Q372436', 'Q38239859',
              'Q4991371', 'Q193391', 'Q4479442', 'Q499165', 'Q2478141', 'Q132050', 'Q170305', 'Q371909', 'Q335954', 'Q851436' ,
              'Q189290', 'Q19643', 'Q215536']
hist_geo = ['Q201788', 'Q67102720', 'Q901402','Q11900058', 'Q12038843', 'Q18524037', 'Q816349', 'Q12347522', 'Q12803959']

phil_scientist = [ 'Q4964182', 'Q11063', 'Q170790' , 'Q16031530' 'Q42973', 'Q14915627', 'Q169470', 'Q155647', 'Q294126',
             'Q21772571', 'Q81096', 'Q2374149']
poet = [ 'Q49757', 'Q16868675' ]

#the group writer is defined to help with the categorization process, but it is not kept in the final categorization
writer = ['Q36180', 'Q864380', 'Q12859263', 'Q482980', 'Q185351', 'Q12144794', 'Q3606216', 'Q214917']

In [346]:
def categorize_occupations(value_1, is_author_index):

    #excluding not authors
    if is_author_index != True:
        return pd.Series({
            'is_politician': None, 'is_geohistorian': None,
            'is_philoscientist': None, 'is_writer': None, 'is_poet': None
        })

    #labeling missing values as unspec
    if pd.isna(value_1):
        return pd.Series({
            'is_politician': 'unspec', 'is_geohistorian': 'unspec', 
            'is_philoscientist': 'unspec', 'is_writer': 'unspec', 'is_poet': 'unspec'
        })
        
   #creating list of occupation
    occ_list =  [q.strip() for q in str(value_1).split(';')] 

    #returns a boolean value for each category
    return pd.Series({
        'is_politician': any(q in occ_list for q in politician),
        'is_geohistorian': any(q in occ_list for q in hist_geo),
        'is_philoscientist': any(q in occ_list for q in phil_scientist),
        'is_writer': any(q in occ_list for q in writer),
        'is_poet': any(q in occ_list for q in poet),
    })
    
df_bool_cols = language_df.apply( #applies the function row per row, lambda extracts the values per row
    lambda row: categorize_occupations(row['occupation_id'], row['is_author_index']),
    axis=1)

df_boolean_occ = pd.concat([language_df, df_bool_cols], axis=1) #axis = 1 means it concats horizontally
df_boolean_occ.to_csv('prova.csv')

In [315]:
def assign_unique_occupations(geohistorian_value, philoscientist_value, writer_value, poet_value, is_author_index, qid):

    if is_author_index != True:
        return None

        
    #defining the group
    poet_mask = poet_value and writer_value and not geohistorian_value and not philoscientist_value
    polymath_mask = writer_value and geohistorian_value and philoscientist_value
    geohistorian_mask = geohistorian_value and not philoscientist_value
    philoscientist_mask = not geohistorian_value and philoscientist_value
    politician_mask = not geohistorian_value and not philoscientist_value and not writer_value


     #manually assigned categories
    if qid in ('Q189971', 'Q3333843', 'Q48174', 'Q11907042', 'Q3614494'):
        return 'geohistorian'
    elif qid in ('Q503075', 'Q159905', 'Q352621'):
        return 'polymath'
    elif qid in ('Q123058247', 'Q112072524'):
        return 'philoscientist'

    
    elif poet_mask:
        return 'poet'
    elif polymath_mask:
        return 'polymath'
    elif geohistorian_mask:
        return 'geohistorian'
    elif philoscientist_mask:
        return 'philoscientist'
    elif politician_mask:
        return 'politician'
    else:
        return None

df_boolean_occ['occupation_category'] = df_boolean_occ.apply(
    lambda row: assign_unique_occupations(
        row['is_geohistorian'], row['is_philoscientist'], row['is_writer'], row['is_poet'],
        row['is_author_index'], row['wikidata_qid']
    ),
    axis=1
)

df_category = df_boolean_occ.drop(columns = ['is_geohistorian', 'is_philoscientist', 'is_writer', 'is_poet'])
print(df_category['occupation_category'].value_counts())

occupation_category
geohistorian      275
polymath           73
politician         38
philoscientist     36
poet               25
Name: count, dtype: int64


## Chronology

We focus only on authors and we model the dates extracted from de Graaf's dataset

The code commented with ## was used to find the authors whose lifespan fell into more than one period, with a small differences of year. These were checked and then manually assigned to the groups (see first part of code)

In [318]:
period_df = df_category.copy()

In [319]:
START_OFFSET = 20  # not productive years

#defining th periods
periods = {
    'archaic': (-1000, -480),
    'classic': (-480, -323),
    'hellenistic': (-323, -146),
    'late_republic': (-146, -27),
    'augustean': (-27, 14),
    'early_empire': (14, 100)}

#manually defined authors
hellenistic_authors = ['Q22712803', 'Q2759424', 'Q3614494', 'Q11907042', 'Q3635627', 'Q2904189', 'Q11916494', 'Q123058247',
                      'Q105641767', 'Q112072524', 'Q88976791', 
                       'Q1465859', 'Q137782564' , 'Q447476', 'Q189971',
                      'Q312548', 'Q16190879', 'Q109567661',
                       'Q62571419' ] #added also Onesicritus, Pytheas, Megasthenes, Diognetus, Baeton + Sagdodonacus even if he is not author

late_republic_authors = ['Q67135690', 'Q11955354' , 'Q550603', 'Q159905'] #added also Frugi, added Hipparchus

early_empire_authors = ['Q759760', 'Q2333984' ]

augustean_authors = ['Q48174', 'Q1351420', 'Q1325527'] #Isidorus, added agrippa and turranius gracilis

def assign_period_final(row):
    start = row['begin_date_final']
    end = row['end_date_final']
    author = row['is_author_index']
    human = row['is_human']
    qid = row['wikidata_qid']
    MARGIN_THRESHOLD = 5

    if human == True:
        if qid in hellenistic_authors:
            return 'hellenistic'
        elif qid in late_republic_authors:
            return 'late_republic'
        elif qid in early_empire_authors:
            return 'early_empire'
        elif qid in augustean_authors:
            return 'augustean'
        elif qid == 'Q137794536':  # aristides
            return 'classic'
        elif qid == 'Q388333':
            return 'unknown'
        elif qid == 'Q296478':  # hanno the navigator
            return 'archaic'

        if pd.isna(start) or pd.isna(end):
            return 'unknown'

        # undatability check on the original dates
        if (end - start) >= 200:
            return "undatable"

        # applying the offset only to calculate the overlap between periods
        start_adjusted = start + START_OFFSET
        # start adjusted must not be higher than the end.
        if start_adjusted >= end:
            start_adjusted = start

        overlap_max = float('-inf')
        overlap_second = float('-inf')
        assigned_period = 'unknown'

        for period, (start_period, end_period) in periods.items():
            #defining the overlap: the nearest end mins the oldest start 
            overlap = min(end, end_period) - max(start_adjusted, start_period)
            if overlap > overlap_max:
                ## overlap_second = overlap_max
             
                overlap_max = overlap # keeps track of the highest overlap
                assigned_period = period
            ## elif overlap > overlap_second:
            ##         overlap_second = overlap

        ## if author == True:
            
        ##     if (overlap_max - overlap_second) <= MARGIN_THRESHOLD:
        ##         assigned_period = "to_be_checked"

        return assigned_period

period_df['period_final'] = period_df.apply(assign_period_final, axis=1)
print(period_df['period_final'].value_counts())

period_final
late_republic    229
hellenistic      206
augustean        134
classic          118
early_empire     107
archaic           68
unknown           17
undatable          2
Name: count, dtype: int64


## Saving 

We save the final file.

In [320]:
period_df.to_csv('nh_categorized.csv', index=False)